# Task 1.1 Baseline: Base Model on GSM8K

In [101]:
import re
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# System prompt: match GSM8K training format (step-by-step with <<...>>, end with #### <number>).

SYSTEM_PROMPT = """
You are a math tutor. 
Solve the problem inside <reasoning> tags, then provide the final integer after "####".

EXAMPLE:

Problem: 12 + 7

Response:

<reasoning>
1. Start with 12.
2. Add 7.
3. 12 + 7 = 19.
<reasoning>
#### 19

It is CRUCIAL that you follow the format EXACTLY. The answer must be provided in the format "#### <integer>" ONLY. 
Please double-check that the format of the answer is "#### <integer>" ONLY. 

"""


In [102]:
# ── Answer extraction ──
# Primary: #### <number> (GSM8K format). Fallback: Answer: <number>, then \boxed{...}.

def extract_boxed(text):
    """Extract content from the last \\boxed{...}, handling nested braces."""
    if not text or "\\boxed{" not in text:
        return None
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    start += len("\\boxed{")
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    if depth == 0:
        return text[start : i - 1].strip()
    return None


def extract_ground_truth(raw_answer):
    """Extract ground-truth answer from GSM8K answer field (ends with #### <number>)."""
    m = re.search(r"####\s*(.+)", raw_answer)
    if m:
        return normalize_answer(m.group(1).strip())
    return normalize_answer(raw_answer.strip())


def normalize_answer(s):
    """Strip and remove commas for numeric comparison."""
    return s.strip().replace(",", "").strip()


def answers_match(pred, gt):
    """Compare predicted and ground-truth answers (both normalized). GSM8K answers are integers."""
    if pred is None or gt is None:
        return False
    pred, gt = normalize_answer(pred), normalize_answer(gt)
    try:
        return int(float(pred)) == int(float(gt))
    except ValueError:
        return pred == gt


def extract_model_answer(text):
    """Extract final answer: prefer #### <number> (GSM8K format), then 'Answer: <number>', then \\boxed{...}."""
    if not text:
        return None
    m = re.search(r"####\s*(.+?)(?:\n|$)", text)
    if m:
        return normalize_answer(m.group(1).strip())
    matches = list(re.finditer(r"Answer:\s*([^\s\n]+)", text, re.IGNORECASE))
    if matches:
        return normalize_answer(matches[-1].group(1))
    boxed = extract_boxed(text)
    if boxed is not None:
        return normalize_answer(boxed)
    return None

**Why does the model sometimes use `\boxed{}` instead of "Answer:"?**  
The base model (Qwen2.5-1.5B-Instruct) was trained on a lot of math and reasoning data where final answers are often written as `\boxed{...}` (e.g. from competition math and papers). That format is a strong prior, so the model may use it even when the system prompt asks for "Answer: &lt;number&gt;". We still prefer "Answer:" in the prompt for consistency, but the extractor falls back to `\boxed{...}` so we don't count correct answers as wrong when the model uses that format.

In [103]:
# Load GSM8K test set and take first 100 questions (fixed subset for all experiments)
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
num_test = 100
test_questions = [gsm8k[i]["question"] for i in range(num_test)]
test_ground_truth = [extract_ground_truth(gsm8k[i]["answer"]) for i in range(num_test)]
print(f"Loaded {num_test} test questions. Example GT: {test_ground_truth[0]!r}")

Loaded 100 test questions. Example GT: '18'


In [104]:
# ── Model loading and batched generation ──

def load_model(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    model.eval()
    print(f"Loaded: {model_name}")
    return model, tokenizer


def build_prompts(tokenizer, questions):
    """Build chat-formatted prompt strings for a list of questions."""
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    return prompts


def generate_batch(model, tokenizer, questions, max_new_tokens=2048):
    """Generate responses for a batch of questions."""
    prompts = build_prompts(tokenizer, questions)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_len:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses

In [105]:
def evaluate_gsm8k(model, tokenizer, questions, ground_truths, batch_size=16):
    """Zero-shot eval on GSM8K. Returns accuracy and list of (pred, gt, correct, raw_response)."""
    correct = 0
    records = []
    num_samples = len(questions)
    for start in tqdm(range(0, num_samples, batch_size), desc="Evaluating"):
        end = min(start + batch_size, num_samples)
        batch_q = questions[start:end]
        batch_gt = ground_truths[start:end]
        responses = generate_batch(model, tokenizer, batch_q)
        for i, (resp, gt) in enumerate(zip(responses, batch_gt)):
            pred = extract_model_answer(resp)
            is_correct = answers_match(pred, gt)
            if is_correct:
                correct += 1
            records.append({"pred": pred, "gt": gt, "correct": is_correct, "response": resp})
    accuracy = correct / num_samples
    return accuracy, records

In [106]:
# Run baseline evaluation (QUESTION 1)
model, tokenizer = load_model()
accuracy, records = evaluate_gsm8k(
    model, tokenizer, test_questions, test_ground_truth, batch_size=16
)
print(f"\nBase model accuracy on 100 GSM8K test questions: {accuracy:.2%}")
print(f"Correct: {sum(r['correct'] for r in records)} / {len(records)}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


Evaluating: 100%|██████████| 7/7 [00:57<00:00,  8.19s/it]


Base model accuracy on 100 GSM8K test questions: 36.00%
Correct: 36 / 100


## QUESTION 2: Inspect 3 Incorrect Cases

In [107]:
# Get indices of incorrect predictions (run after Q1 evaluation)
incorrect_indices = [i for i, r in enumerate(records) if not r["correct"]]
print(f"Total incorrect: {len(incorrect_indices)}. Inspecting first 3.\n")

# Show 3 failure cases
num_inspect = min(3, len(incorrect_indices))
for case_num, idx in enumerate(incorrect_indices[:num_inspect], 1):
    q = test_questions[idx]
    r = records[idx]
    print("=" * 80)
    print(f"FAILURE CASE {case_num}")
    print("=" * 80)
    print("\nQuestion:")
    print(q)
    print("\nModel's solution (excerpt):")
    # Show first ~600 chars; full response available as r["response"] if needed
    excerpt = r["response"]
    print(excerpt)
    print("\nExtracted answer:", repr(r["pred"]))
    print("Ground-truth answer:", repr(r["gt"]))
    print()

Total incorrect: 64. Inspecting first 3.

FAILURE CASE 1

Question:
Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?

Model's solution (excerpt):
<reasoning>
1. First, calculate the total cost of buying the house including repairs:
   - Cost of the house: $80,000
   - Amount spent on repairs: $50,000
   - Total cost: $80,000 + $50,000 = $130,000

2. Next, determine the increase in the value of the house due to the renovations:
   - Increase in value: 150% of the original price
   - Value increase amount: 150/100 * $80,000 = $120,000

3. Calculate the new value of the house after the renovations:
   - New value: Original price + Value increase
   - New value: $80,000 + $120,000 = $200,000

4. Finally, find the profit made by subtracting the total cost from the new value:
   - Profit: New value - Total cost
   - Profit: $200,000 - $130,000 = $170,000
</reas

### Failure mode classification and patterns

**Case 1 – Failure mode:** Arithmetic slip  

**Case 2 – Failure mode:** Arithmetic slip  

**Case 3 – Failure mode:** Misunderstanding the problem (and reasoning error)  

**Recurring patterns:** arithmetic slips or misunderstanding the problem

## QUESTION 3: LoRA Hyperparameters 

### 1. LoRA rank (r)

**What it controls:** LoRA adapts a weight matrix $W$ by a low-rank update $W' = W + B A$, where $B$ is $(d \times r)$ and $A$ is $(r \times k)$. The rank $r$ directly controls the capacity of the adapter: how many independent "directions" the model can use to adapt each layer.

**If you increase $r$:**  
- **Pros:** More capacity to learn task-specific patterns; can improve accuracy and fit harder tasks.  
- **Cons:** higher memory compute per step; longer training; higher overfitting risk.

**If you decrease $r$:**  
- **Pros:** lower memory and compute; faster steps; less overfitting. 
- **Cons:** may underfit on complex tasks; model might not be able to capture all needed adaptations.

---

### 2. LoRA alpha

**What it controls:** The LoRA forward pass is typically $y = Wx + (\alpha/r)\, (BA)x$. So the adapter's output is scaled by $\alpha/r$. For a fixed $r$, $\alpha$ controls the influence of the adapter relative to the frozen base model. 

**If you increase $\alpha$:**  
- **Pros:** Stronger effective learning signal from the adapter; can speed up adaptation and make the fine-tuned behavior more pronounced.  
- **Cons:** Can make training less stable (larger effective step size); in extreme cases the adapter can dominate and hurt generalization.

**If you decrease $\alpha$:**  
- **Pros:** **More stable** training; adapter changes are gentler.  
- **Cons:** **slower** adaptation; may need more epochs or risk underfitting if the model barely changes.

---

### 3. Gradient accumulation

**What it controls:** Number of mini-batch forward/backward passes before each optimizer step;

**If you increase gradient accumulation:**  
- **Pros:** Larger effective batch size; more stable gradient estimates; often smoother loss and more predictable training;
- **Cons:** more time and compute; can generalize slightly worse.

**If you decrease gradient accumulation:**  
- **Pros:** Fewer passes per step; smaller effective batch.  
- **Cons:** less stable training, may need lower learning rate or more careful tuning

## QUESTION 4: Parameter counts

In [2]:
# Q4: Base model and LoRA parameter counts (default: r=8, alpha=16, attention only q,k,v,o)
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
total_params = sum(p.numel() for p in model.parameters())

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
pct = 100.0 * trainable_params / total_params

print("(a) Total parameters (base model):", f"{total_params:,}")
print("(b) Trainable LoRA parameters:", f"{trainable_params:,}")
print("(c) Percentage of parameters trained:", f"{pct:.2f}%")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

(a) Total parameters (base model): 1,543,714,304
(b) Trainable LoRA parameters: 2,179,072
(c) Percentage of parameters trained: 0.14%


**Why the percentage is small:** We only train the LoRA adapter matrices, not the full weight matrices. The base model (1.5B params) is frozen.

**How LoRA achieves this:** For each targeted linear layer $W$, LoRA adds a low-rank update $W' = W + B A$ with $B$ (d X r) and $A$ (r X d). So instead of training $d^2$ parameters per layer, we train $2rd$ (with $r=8$). Only the attention projections are adapted, and with small $r$ the total trainable parameters are on the order of a few million instead of billions- hence a small fraction of the full model.

## QUESTION 5: LoRA SFT using 1,000 Training Examples

In [7]:
# Q5: Load 1,000 train examples, build SFT dataset, train LoRA with default hyperparameters (no trl)
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_SEQ_LENGTH = 1024

# Load 1,000 training examples
gsm8k_train = load_dataset("openai/gsm8k", "main", split="train").select(range(1000))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # for training

def tokenize_sft(ex):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["question"]},
        {"role": "assistant", "content": ex["answer"]},
    ]
    prompt_messages = messages[:2]  # system + user
    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    full_ids = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_SEQ_LENGTH).input_ids
    prompt_len = len(prompt_ids)
    if prompt_len >= len(full_ids):
        labels = full_ids  # no prompt masking fallback
    else:
        labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {"input_ids": full_ids, "labels": labels, "attention_mask": [1] * len(full_ids)}

train_dataset = gsm8k_train.map(
    tokenize_sft,
    remove_columns=gsm8k_train.column_names,
    desc="Tokenizing",
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="./q5_lora_gsm8k",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    num_train_epochs=1,
    bf16=True,
    logging_steps=20,
    save_strategy="epoch",
)

def collate_fn(batch):
    pad_id = tokenizer.pad_token_id
    max_len = max(len(b["input_ids"]) for b in batch)
    max_len = min(max_len, MAX_SEQ_LENGTH)
    input_ids = []
    labels = []
    attention_mask = []
    for b in batch:
        ids = b["input_ids"][:max_len]
        lbls = b["labels"][:max_len]
        pad_len = max_len - len(ids)
        input_ids.append(ids + [pad_id] * pad_len)
        labels.append(lbls + [-100] * pad_len)
        attention_mask.append([1] * len(ids) + [0] * pad_len)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=collate_fn,
)
trainer.train()
trainer.save_model("./q5_lora_gsm8k")
tokenizer.save_pretrained("./q5_lora_gsm8k")
print("LoRA SFT training done. Adapter saved to ./q5_lora_gsm8k")

Tokenizing:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
20,0.518054


LoRA SFT training done. Adapter saved to ./q5_lora_gsm8k


In [108]:
# Q5: Evaluate trained LoRA on the same 100 GSM8K test questions as Q1
# (Run the data-load and evaluate_gsm8k cells above first, or run the notebook from the start.)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "./q5_lora_gsm8k"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager",
)
model = PeftModel.from_pretrained(model, adapter_path)
model.eval()

accuracy, records_q5 = evaluate_gsm8k(
    model, tokenizer, test_questions, test_ground_truth, batch_size=16
)
print(f"\nQ5 LoRA SFT accuracy on 100 GSM8K test questions: {accuracy:.2%}")
print(f"Correct: {sum(r['correct'] for r in records_q5)} / {len(records_q5)}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating: 100%|██████████| 7/7 [01:04<00:00,  9.18s/it]


Q5 LoRA SFT accuracy on 100 GSM8K test questions: 46.00%
Correct: 46 / 100
